In [1]:
import os
import glob
import pandas as pd

ptdfs_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\PTDFs"
limits_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\Limits"

# get all CSV files
ptdfs_csv_files = glob.glob(os.path.join(ptdfs_path, "*.csv"))
limits_csv_files = glob.glob(os.path.join(limits_path,"*.csv"))

ptdf_dfs = {}
limits_dfs = {}

for file in ptdfs_csv_files:
    # name without extension → key
    name = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file, sep=";", decimal=".")
    ptdf_dfs[name] = df

for file in limits_csv_files:
    name = os.path.splitext(os.path.basename(file))[0]
    df = pd.read_csv(file, sep = ";", decimal = ".")
    limits_dfs[name] = df

print(f"Loaded {len(ptdf_dfs)} ptdfs files and {len(limits_dfs)} limits")
print(list(ptdf_dfs.keys()))
print(list(limits_dfs.keys()))

Loaded 40 ptdfs files and 2 limits
['Distribution_Factors_Results_(SYM)_Base', 'Distribution_Factors_Results_(SYM)_LODF', 'Line_01__02_Cnt', 'Line_01__39_Cnt', 'Line_02__03_Cnt', 'Line_02__25_Cnt', 'Line_03__04_Cnt', 'Line_03__18_Cnt', 'Line_04__05_Cnt', 'Line_04__14_Cnt', 'Line_05__06_Cnt', 'Line_05__08_Cnt', 'Line_06__07_Cnt', 'Line_06__11_Cnt', 'Line_07__08_Cnt', 'Line_08__09_Cnt', 'Line_09__39_Cnt', 'Line_10__11_Cnt', 'Line_10__13_Cnt', 'Line_13__14_Cnt', 'Line_14__15_Cnt', 'Line_15__16_Cnt', 'Line_16__17_Cnt', 'Line_16__19_Cnt', 'Line_16__21_Cnt', 'Line_16__24_Cnt', 'Line_17__18_Cnt', 'Line_17__27_Cnt', 'Line_21__22_Cnt', 'Line_22__23_Cnt', 'Line_23__24_Cnt', 'Line_25__26_Cnt', 'Line_26__27_Cnt', 'Line_26__28_Cnt', 'Line_26__29_Cnt', 'Line_28__29_Cnt', 'Trf_06__31_Cnt', 'Trf_11__12_Cnt', 'Trf_13__12_Cnt', 'Trf_19__20_Cnt']
['limits_lines', 'limits_trafos']


In [16]:
def load_csv_folder(folder_path, sep=";", decimal="."):
    """
    Load all CSV files from a folder into a dictionary of DataFrames.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing CSV files.
    sep : str, optional
        CSV separator, by default ";"
    decimal : str, optional
        Decimal separator, by default "."

    Returns
    -------
    dict
        Dictionary where:
        - key = file name without extension
        - value = pandas DataFrame
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = {}

    for file in csv_files:
        name = os.path.splitext(os.path.basename(file))[0]
        df = pd.read_csv(file, sep=sep, decimal=decimal)
        dfs[name] = df

    return dfs

def filter_ptdf_columns(df):
    """
    This function receives a ptdf result dataframe from powerfactory and clean the columns
    It returns a cleaned dataframe with only the ptdf for lines and trafos of the 
    original dataframe 
    """
    df.rename(columns={df.columns[0]:"Injection Bus"}, inplace=True)

    first_col = df.columns[0] # Keep the bus index
    cols = [first_col]
    seen_elements = set()
    df = df.iloc[1:].reset_index(drop = True)
    for c in df.columns[1:]:
        # keep only lines and trafos
        if not (c.startswith("Line") or c.startswith("Trf")):
            continue

        # remove .1, .2 etc
        if any(f".{i}" in c for i in range(1, 10)):
            continue

        # avoid duplicates (just in case)
        if c not in seen_elements:
            cols.append(c)
            seen_elements.add(c)

    return df[cols]

In [ ]:
ptdf_dfs = load_csv_folder(ptdfs_path)
limits_dfs = load_csv_folder(limits_path)

ptdf_dfs_filtered = {}
for name,df in ptdf_dfs.items():
    filtered_df = filter_ptdf_columns(df)
    filtered_df = filtered_df.reset_index(drop = True)
    ptdf_dfs_filtered[name] = filtered_df


In [2]:
#dfs["Distribution_Factors_Results_(SYM)_Base"].head()
# for name,df in ptdf_dfs.items():
#     print(name,df.shape)

for name, df in limits_dfs.items():
    print(name,df.shape)

limits_lines (34, 7)
limits_trafos (12, 5)


In [3]:
limits_dfs["limits_lines"].head()

,name,uline,iline,smax,loading,p_used,margin
0,Line 01 - 02,345.0,1.0,597.557529,21.047809,125.772767,471.784762
1,Line 01 - 39,345.0,1.0,597.557529,26.377798,157.622518,439.935011
2,Line 02 - 03,345.0,1.0,597.557529,61.106350,365.145594,232.411935
3,Line 02 - 25,345.0,1.0,597.557529,41.252467,246.507222,351.050306
4,Line 03 - 04,345.0,1.0,597.557529,26.437980,157.982139,439.575389


In [4]:
### I need to correct the structure, so now i am "losing" the first row, where it shows where the bus injection was (which bus had the injection)
# Then I need to reshape the ptdf frame and then merge by element name

# I also need to clean all the ptdfs results meaning having a clean dictionary of ptdfs 

In [5]:

def filter_ptdf_columns(df):
    """
    This function receives a ptdf result dataframe from powerfactory and clean the columns
    It returns a cleaned dataframe with only the ptdf for lines and trafos of the 
    original dataframe 
    """
    df.rename(columns={df.columns[0]:"Injection Bus"}, inplace=True)

    first_col = df.columns[0] # Keep the bus index
    cols = [first_col]
    seen_elements = set()
    df = df.iloc[1:].reset_index(drop = True)
    for c in df.columns[1:]:
        # keep only lines and trafos
        if not (c.startswith("Line") or c.startswith("Trf")):
            continue

        # remove .1, .2 etc
        if any(f".{i}" in c for i in range(1, 10)):
            continue

        # avoid duplicates (just in case)
        if c not in seen_elements:
            cols.append(c)
            seen_elements.add(c)

    return df[cols]
        

In [6]:
ptdfs_cleaned = {}
for name,df in ptdf_dfs.items():
    df = filter_ptdf_columns(df)
    df = df.reset_index(drop = True)
    ptdfs_cleaned[name] = df

ptdfs_cleaned["Line_01__02_Cnt"].head()

,Injection Bus,Line 01 - 02,Line 01 - 39,Line 02 - 03,Line 02 - 25,Line 03 - 04,Line 03 - 18,Line 04 - 05,Line 04 - 14,Line 05 - 06,...,Trf 10 - 32,Trf 11 - 12,Trf 13 - 12,Trf 19 - 20,Trf 19 - 33,Trf 20 - 34,Trf 22 - 35,Trf 23 - 36,Trf 25 - 37,Trf 29 - 38
0,1.000000,----,1.000000,-0.001261,0.001261,-0.003883,0.002740,-0.025913,0.022184,0.484048,...,----,-0.002519,0.002518,-0.000020,-0.000022,-0.000019,----,-0.000008,-0.000002,-0.000004
1,2.000000,----,----,0.805862,0.194138,0.679256,0.117586,0.636324,0.040056,0.577457,...,----,-0.031560,0.031575,0.000009,0.000010,0.000009,----,0.000004,0.000005,0.000007
2,3.000000,----,----,-0.055336,0.055336,0.717534,0.227821,0.649469,0.064941,0.589338,...,----,-0.031080,0.031094,-0.000010,-0.000011,-0.000009,----,-0.000004,-0.000001,-0.000001
3,4.000000,----,----,-0.008632,0.008632,-0.041900,0.033463,0.709720,0.248684,0.643941,...,----,-0.026024,0.026035,-0.000019,-0.000020,-0.000017,----,-0.000007,-0.000002,-0.000003
4,5.000000,----,----,-0.001625,0.001625,-0.006648,0.005109,-0.045495,0.038964,0.865250,...,----,-0.004192,0.004192,-0.000013,-0.000015,-0.000012,----,-0.000005,-0.000002,-0.000002


In [7]:
df = ptdfs_cleaned["Line_01__02_Cnt"]

df_long = df.melt(
    id_vars="Injection Bus",
    var_name="Element",
    value_name="PTDF"
)

df_long.head()


,Injection Bus,Element,PTDF
0,1.000000,Line 01 - 02,----
1,2.000000,Line 01 - 02,----
2,3.000000,Line 01 - 02,----
3,4.000000,Line 01 - 02,----
4,5.000000,Line 01 - 02,----


In [8]:
limit_lines = limits_dfs["limits_lines"].copy()
limit_trafos = limits_dfs["limits_trafos"].copy()

limit_lines = limit_lines.rename(columns={"name":"Element"})
limit_trafos = limit_trafos.rename(columns={"name":"Element"})

limit_elements = pd.concat([limit_lines,limit_trafos],ignore_index= True)

In [9]:
result = df_long.merge(limit_elements, on = "Element", how = "left")
result["Injection Bus"] = result["Injection Bus"].astype(float).astype(int)
result = result.sort_values(by =["Injection Bus", "Element"])
result = result.reset_index(drop = True)
result.head()

,Injection Bus,Element,PTDF,uline,iline,smax,loading,p_used,margin
0,1,Line 01 - 02,----,345.0,1.0,597.557529,21.047809,125.772767,471.784762
1,1,Line 01 - 39,1.000000,345.0,1.0,597.557529,26.377798,157.622518,439.935011
2,1,Line 02 - 03,-0.001261,345.0,1.0,597.557529,61.106350,365.145594,232.411935
3,1,Line 02 - 25,0.001261,345.0,1.0,597.557529,41.252467,246.507222,351.050306
4,1,Line 03 - 04,-0.003883,345.0,1.0,597.557529,26.437980,157.982139,439.575389


In [10]:
# Now operate the results
import numpy as np
result["PTDF"] = pd.to_numeric(result["PTDF"], errors="coerce")
result["PTDF"] = result["PTDF"].fillna(0)
result["Transfer Limit"] = np.where(
    result["PTDF"] == 0,
    np.inf,  # no limitation from this element
    (result["smax"] - result["p_used"]) / result["PTDF"].abs()
)
result.head()

,Injection Bus,Element,PTDF,uline,iline,smax,loading,p_used,margin,Transfer Limit
0,1,Line 01 - 02,0.000000,345.0,1.0,597.557529,21.047809,125.772767,471.784762,inf
1,1,Line 01 - 39,1.000000,345.0,1.0,597.557529,26.377798,157.622518,439.935011,4.399350e+02
2,1,Line 02 - 03,-0.001261,345.0,1.0,597.557529,61.106350,365.145594,232.411935,1.843076e+05
3,1,Line 02 - 25,0.001261,345.0,1.0,597.557529,41.252467,246.507222,351.050306,2.783904e+05
4,1,Line 03 - 04,-0.003883,345.0,1.0,597.557529,26.437980,157.982139,439.575389,1.132051e+05


# Logic of the code

1. Read the CSV results from Powerfactory into a dictionary of dataframes : raw_dfs_ptdfs
2. Read the CSV limit results from Powerfactory : raw_dfs_limts
3. Filter the columns of the raw Dataframe of PTDFS to extract only the columns of interest : dfs_ptdfs_filtered
4. Combine the limits into one single Dataframe of limits: dfs_limits_combined
5. Reshape the dfs_ptdfs_filtered dictionary, melt function, transform from wide to long format, id: injection bus, variable: Element, value: PTDF (create function)
6. Rename the dfs_limits_combined dataframe column: "name" to "Element" so merging is possible
7. Merge on element, how: left -> as a result the dictionary of results is created 
8. Transform the relevant columns into the correct type of variable so operating is possible, do it  for each df in the dict of dfs
9. Calculate the transfer limit of each element, for each bus, for each case
10. Order the results to observe the max to the less impacted elements for each bus/case
11. Present the table results